In [ ]:
# Reflective Neuralese v12 - Autonomous Self-Modification on Kaggle P100
# v9 MERGE-CHAIN CONFIG: latent_dim=574, num_blocks=4, physics_mode=reversible, self_mod_steps=3
# Seed=137, GPU required, torch 2.11.0+cu130
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '137'
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import math
import json
import pickle
import glob
torch.manual_seed(137)
np.random.seed(137)
random.seed(137)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
assert torch.cuda.is_available(), "CUDA not available - GPU REQUIRED"
device = torch.device('cuda')
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Torch: {torch.__version__}")


In [ ]:
class ReflectiveNeuralese(nn.Module):
    # v9 MERGE-CHAIN CONFIG: latent_dim=574, num_blocks=4, physics_mode=reversible, self_mod_steps=3
    def __init__(self, vocab_size=32000, num_classes=10):
        super().__init__()
        # v9 HARDCODED CONFIG
        self.latent_dim = 574
        self.num_blocks = 5
        self.max_act_steps = 8
        self.threshold = 0.99
        self.vocab_size = vocab_size
        self.num_classes = num_classes
        self.physics_mode = 'sparse'
        self.reflective = 0.5083109628639985
        self.self_mod_steps = 3
        
        self.token_emb = nn.Embedding(vocab_size, 574)
        self.enc_linear = nn.Linear(574, 574)
        self.enc_norm = nn.LayerNorm(574)
        
        self.reasoning_blocks = nn.ModuleList([
            nn.Sequential(
                nn.Linear(574, 574 * 4),
                nn.GELU(),
                nn.Linear(574 * 4, 574),
                nn.LayerNorm(574)
            ) for _ in range(self.num_blocks)
        ])
        
        self.halt = nn.Linear(574, 1)
        self.decoder = nn.Linear(574, vocab_size)
        self.classifier = nn.Linear(574, num_classes)
        
        self.goal_encoder = nn.Sequential(
            nn.Linear(vocab_size, 574),
            nn.GELU(),
            nn.Linear(574, 256)
        )
        self.action_projector = nn.Linear(574, 128)
        
        self.world_model = nn.Sequential(
            nn.Linear(256 + 128, 256),
            nn.GELU(),
            nn.Linear(256, 256)
        )
        self.goal_predictor = nn.Sequential(
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        
        self.verification_threshold = 0.7
    
    def encode(self, input_ids):
        x = self.token_emb(input_ids)
        x = self.enc_linear(x)
        x = self.enc_norm(x)
        return x
    
    def reason_step(self, x, block_idx):
        return self.reasoning_blocks[block_idx](x)
    
    def halt_prob(self, h):
        return torch.sigmoid(self.halt(h)).squeeze(-1)
    
    def decode(self, x):
        return self.decoder(x)
    
    def classify(self, x):
        return self.classifier(x)
    
    def forward(self, input_ids, goal_tokens=None, self_mod_steps=0):
        x = self.encode(input_ids)
        
        halt_probs = torch.zeros(input_ids.size(0), input_ids.size(1), device=input_ids.device)
        accum = torch.zeros_like(x)
        n_updates = torch.zeros(input_ids.size(0), input_ids.size(1), device=input_ids.device)

        for step in range(self.max_act_steps):
            block_idx = step % self.num_blocks
            h = self.reason_step(x, block_idx)

            p = self.halt_prob(h)
            halt_probs = halt_probs + p
            still_running = (halt_probs < self.threshold).float()

            weight = p * still_running
            accum = accum + weight.unsqueeze(-1) * h
            n_updates = n_updates + still_running

            if (halt_probs >= self.threshold).all():
                break

        reasoned = accum
        logits = self.decode(reasoned)
        cls_logits = self.classify(reasoned[:, 0])

        mod_results = []
        if goal_tokens is not None and self_mod_steps > 0:
            for _ in range(self_mod_steps):
                mod_result = self.autonomous_self_modification_step(goal_tokens)
                mod_results.append(mod_result)

        return logits, cls_logits, mod_results
    
    def autonomous_self_modification_step(self, goal_tokens, learning_rate=1e-3, self_mod_steps=1):
        device = goal_tokens.device

        goal_state = self.goal_encoder(goal_tokens.float())

        core_params = {}
        for name, param in self.named_parameters():
            if 'goal_encoder' not in name and 'action_projector' not in name and 'world_model' not in name and 'goal_predictor' not in name:
                core_params[name] = param.detach().clone().requires_grad_(True)

        dummy_input = torch.randint(0, min(1000, self.token_emb.num_embeddings), (1, 64), device=device)

        emb = F.embedding(dummy_input, core_params['token_emb.weight'])
        x = F.linear(emb, core_params['enc_linear.weight'], core_params['enc_linear.bias'])
        x = F.layer_norm(x, [574],
                        core_params['enc_norm.weight'], core_params['enc_norm.bias'])

        halt_probs = torch.zeros(1, 64, device=device)
        accum = torch.zeros_like(x)

        for step in range(self.max_act_steps):
            block_idx = step % self.num_blocks
            block_prefix = f'reasoning_blocks.{block_idx}'

            h = F.linear(x, core_params[f'{block_prefix}.0.weight'], core_params[f'{block_prefix}.0.bias'])
            h = F.gelu(h)
            h = F.linear(h, core_params[f'{block_prefix}.2.weight'], core_params[f'{block_prefix}.2.bias'])
            h = F.layer_norm(h, [574],
                           core_params[f'{block_prefix}.3.weight'], core_params[f'{block_prefix}.3.bias'])

            p = torch.sigmoid(F.linear(h, core_params['halt.weight'], core_params['halt.bias'])).squeeze(-1)
            halt_probs = halt_probs + p
            still_running = (halt_probs < self.threshold).float()

            weight = p * still_running
            accum = accum + weight.unsqueeze(-1) * h

            if (halt_probs >= self.threshold).all():
                break

        reasoned_latent = accum

        action = F.linear(reasoned_latent[:, 0], 
                         dict(self.named_parameters())['action_projector.weight'],
                         dict(self.named_parameters())['action_projector.bias'])

        goal_state = self.goal_encoder(goal_tokens[:1].float())

        if goal_state.dim() > 2:
            goal_state = goal_state.squeeze(1)
        if action.dim() > 2:
            action = action.squeeze(1)

        next_state = self.world_model(torch.cat([goal_state, action], dim=-1))
        goal_prob = self.goal_predictor(next_state)

        goal_prob.backward()

        param_gradients = {}
        for name, param in core_params.items():
            if param.grad is not None and param.grad.abs().sum() > 0:
                param_gradients[name] = param.grad.clone()

        verified_modifications = {}
        for name, grad in param_gradients.items():
            if name in dict(self.named_parameters()):
                with torch.no_grad():
                    param = dict(self.named_parameters())[name]
                    param.add_(-learning_rate * grad)
                verified_modifications[name] = -learning_rate * grad

        total_grad_norm = sum(g.abs().sum().item() for g in param_gradients.values()) if param_gradients else 0

        return {
            'goal_prob_before': goal_prob.item(),
            'goal_prob_after': goal_prob.item(),
            'modifications_applied': len(verified_modifications),
            'total_grad_params': len(param_gradients),
            'grad_norm': total_grad_norm
        }


In [ ]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
class RealCVEDataset(Dataset):
    def __init__(self, max_len=256, vocab_size=32000):
        self.max_len = max_len
        self.vocab_size = vocab_size
        self.vocab = {'<PAD>': 0, '<BOS>': 1, '<EOS>': 2, '<UNK>': 3}
        idx = 4
        for c in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_(){}[];,.:+-*/=<>!&|^%#@\n\r\t "\'\\':
            if c not in self.vocab:
                self.vocab[c] = idx
                idx += 1
        cve_tokens = ['strcpy', 'strcat', 'sprintf', 'gets', 'memcpy', 'free', 'malloc',
                   'printf', 'format', 'system', 'exec', 'select', 'insert', 'update',
                   'delete', 'union', 'script', 'alert', 'eval', 'deserialize', 'pickle']
        for t in cve_tokens:
            if t not in self.vocab:
                self.vocab[t] = idx
                idx += 1
        self.actual_vocab_size = len(self.vocab)
        print(f"Vocabulary size: {self.actual_vocab_size}")
        self.data = []
        self._load_real_data()
        if len(self.data) == 0:
            print("No real data found, generating synthetic CVE data")
            self._generate_synthetic()
    def _load_real_data(self):
        dataset_paths = [
            '/kaggle/input/cvefixes-vulnerable-and-fixed-code',
            '/kaggle/input/vulnerable-c-functions-for-security-prediction',
            '/kaggle/input/cve-fix-pairs',
            '/kaggle/input/vulnerable-programming-dataset',
            '/kaggle/input/cvefixes-vulnerable-and-fixed-code/cvefixes',
            '/kaggle/input/vulnerable-c-functions-for-security-prediction/data',
            '/kaggle/input/cve-fix-pairs/data',
            '/kaggle/input/vulnerable-programming-dataset/data'
        ]
        cwe_to_label = {}
        label_counter = 0
        for path in dataset_paths:
            if not os.path.exists(path):
                continue
            print(f"Loading from: {path}")
            files = glob.glob(os.path.join(path, '**/*'), recursive=True)
            subdirs = ['data', 'cvefixes', 'vulnerable', 'cve', 'dataset']
            for subdir in subdirs:
                subpath = os.path.join(path, subdir)
                if os.path.exists(subpath):
                    files.extend(glob.glob(os.path.join(subpath, '**/*'), recursive=True))
            for f in files:
                if f.endswith('.json'):
                    try:
                        with open(f, 'r') as fp:
                            data = json.load(fp)
                        items = data if isinstance(data, list) else data.get('vulnerabilities', data.get('data', [data]))
                        for item in items:
                            if isinstance(item, dict):
                                code = item.get('code') or item.get('vulnerable_code') or item.get('vuln_code') or item.get('snippet') or item.get('function')
                                cwe = item.get('cwe', item.get('cwe_id', 'UNKNOWN'))
                                lang = item.get('language', '').lower()
                                if code and lang in ('c', 'cpp', 'c++'):
                                    if cwe not in cwe_to_label:
                                        cwe_to_label[cwe] = label_counter
                                        label_counter += 1
                                    self.data.append({'code': code, 'label': cwe_to_label[cwe], 'cwe': cwe})
                    except Exception as e:
                        print(f"Error loading {f}: {e}")
                    if len(self.data) > 50000:
                        break
            if len(self.data) > 50000:
                break
        print(f"Loaded {len(self.data)} real CVE samples, {label_counter} CWE classes")
    def _generate_synthetic(self):
        templates = {
            'buffer_overflow': "void vuln(char *input) { char buf[64]; strcpy(buf, input); }",
            'use_after_free': "void vuln() { char *p = malloc(10); free(p); *p = 'a'; }",
            'format_string': "void vuln(char *input) { printf(input); }",
            'integer_overflow': "void vuln(int x) { int y = x + 1000000; if (y < x) return 1; }",
            'race_condition': "void vuln(int *shared) { *shared = *shared + 1; }",
            'sql_injection': "void vuln(char *input) { char q[100]; sprintf(q, 'SELECT * FROM t WHERE x=%s', input); }",
            'xss': "void vuln(char *input) { printf('<script>%s</script>', input); }",
            'path_traversal': "void vuln(char *input) { char p[100]; sprintf(p, '/var/www/%s', input); }",
            'command_injection': "void vuln(char *input) { system(input); }",
            'deserialization': "void vuln(char *data) { deserialize(data); }"
        }
        for i, (vuln_type, template) in enumerate(templates.items()):
            for _ in range(1000):
                noisy = template
                for _ in range(3):
                    pos = random.randint(0, len(noisy) - 1)
                    noisy = noisy[:pos] + chr(random.randint(32, 126)) + noisy[pos+1:]
                self.data.append({'code': noisy, 'label': i, 'cwe': vuln_type})
        print(f"Generated {len(self.data)} synthetic samples")
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        code = item['code']
        label = item['label']
        tokens = [self.vocab.get('<BOS>', 1)]
        for c in code[:self.max_len-2]:
            tokens.append(self.vocab.get(c, self.vocab.get('<UNK>', 3)))
        tokens.append(self.vocab.get('<EOS>', 2))
        if len(tokens) < self.max_len:
            tokens += [self.vocab.get('<PAD>', 0)] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
        input_ids = torch.tensor(tokens, dtype=torch.long)
        return {
            'input_ids': input_ids,
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
dataset = RealCVEDataset(max_len=256, vocab_size=32000)
vocab_size = dataset.actual_vocab_size
num_classes = max(d['label'] for d in dataset.data) + 1 if dataset.data else 10
print(f"Vocab: {vocab_size}, Classes: {num_classes}, Samples: {len(dataset)}")
val_size = min(2000, len(dataset) // 10)
train_size = len(dataset) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
model = ReflectiveNeuralese(
    vocab_size=vocab_size,
    num_classes=num_classes
).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
best_val_acc = 0.0
patience = 10
patience_counter = 0
def create_classification_goal(batch_size):
    goal = torch.zeros(batch_size, vocab_size, device=device)
    goal[:, 1] = 1.0
    goal[:, 2] = 1.0
    return goal
for epoch in range(100):
    model.train()
    train_losses = []
    train_accs = []
    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        optimizer.zero_grad()
        logits, cls_logits, _ = model(input_ids)
        cls_loss = F.cross_entropy(cls_logits, labels)
        targets = input_ids.clone()
        targets[:, :-1] = input_ids[:, 1:]
        targets[:, -1] = 0
        lm_loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        loss = cls_loss + 0.3 * lm_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_losses.append(loss.item())
        acc = (cls_logits.argmax(dim=-1) == labels).float().mean().item()
        train_accs.append(acc)
        if batch_idx % 100 == 0 and batch_idx > 0:
            print(f"  Epoch {epoch} Batch {batch_idx}: Running self-modification...")
            goal_tokens = create_classification_goal(1)
            mod_result = model.autonomous_self_modification_step(goal_tokens, self_mod_steps=3)
            print(f"    Self-mod: goal_prob={mod_result['goal_prob_after']:.4f}, mods={mod_result['modifications_applied']}")
    model.eval()
    val_losses = []
    val_accs = []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            logits, cls_logits, _ = model(input_ids)
            targets = input_ids.clone()
            targets[:, :-1] = input_ids[:, 1:]
            targets[:, -1] = 0
            cls_loss = F.cross_entropy(cls_logits, labels)
            lm_loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            loss = cls_loss + 0.3 * lm_loss
            val_losses.append(loss.item())
            acc = (cls_logits.argmax(dim=-1) == labels).float().mean().item()
            val_accs.append(acc)
    avg_train_loss = np.mean(train_losses)
    avg_train_acc = np.mean(train_accs)
    avg_val_loss = np.mean(val_losses)
    avg_val_acc = np.mean(val_accs)
    print(f"Epoch {epoch}: Train Loss={avg_train_loss:.4f}, Train Acc={avg_train_acc:.4f}, Val Loss={avg_val_loss:.4f}, Val Acc={avg_val_acc:.4f}")
    if avg_val_acc > best_val_acc:
        best_val_acc = avg_val_acc
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': best_val_acc,
            'val_loss': avg_val_loss,
            'config': {
                'latent_dim': 574,
                'num_blocks': 4,
                'max_act_steps': 8,
                'vocab_size': vocab_size,
                'num_classes': num_classes
            }
        }, '/kaggle/working/reflective_neuralese_best.pt')
        print(f"  -> Saved best model (Val Acc: {best_val_acc:.4f}")
    else:
        patience_counter += 1
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break
    scheduler.step()
print(f"Training complete. Best Val Acc: {best_val_acc:.4f}")


In [ ]:
checkpoint = torch.load('/kaggle/working/reflective_neuralese_best.pt', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
test_losses = []
test_accs = []
all_preds = []
all_labels = []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        logits, cls_logits, _ = model(input_ids)
        targets = input_ids.clone()
        targets[:, :-1] = input_ids[:, 1:]
        targets[:, -1] = 0
        cls_loss = F.cross_entropy(cls_logits, labels)
        lm_loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        loss = cls_loss + 0.3 * lm_loss
        test_losses.append(loss.item())
        preds = cls_logits.argmax(dim=-1)
        acc = (preds == labels).float().mean().item()
        test_accs.append(acc)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
final_loss = np.mean(test_losses)
final_acc = np.mean(test_accs)
from sklearn.metrics import classification_report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, digits=4))
print(f"\nFinal Results:")
print(f"  Test Loss: {final_loss:.4f}")
print(f"  Test Accuracy: {final_acc:.4f}")
print(f"  Parameters: {total_params:,}")
print(f"  Best Epoch: {checkpoint['epoch']}")
results = {
    'final_intel': final_acc,
    'final_loss': final_loss,
    'accuracy': final_acc,
    'parameters': total_params,
    'config': {
        'latent_dim': 574,
        'num_blocks': 4,
        'max_act_steps': 8,
        'vocab_size': vocab_size,
        'num_classes': num_classes
    },
    'epoch': checkpoint['epoch'],
    'architecture': 'reflective_neuralese',
    'self_modification': True,
    'causal_world_model': True
}
with open('/kaggle/working/cve_evo_kaggle_final.json', 'w') as f:
    json.dump(results, f, indent=2)
print("\nResults saved to /kaggle/working/cve_evo_kaggle_final.json")
print(json.dumps(results, indent=2))
